# Task 2 — Rule-Based Inference Engine
### Smart Electricity Consumption Optimizer

This notebook builds directly on the Task 1 frame-based knowledge model (User, House, Appliance, Weather, Occupancy, Electricity Tariff, Energy Consumption, Recommendation).

It covers:
1. IF–THEN rule formulation
2. Choice of inference mechanism (Forward vs Backward Chaining)
3. Implementation of the rule engine
4. Testing with sample inputs
5. Explanation of the decision path for each test

Just run the cells top to bottom in Colab — each one builds on the last.

## 1. IF–THEN Rules

The rules below encode the constraints from Task 1 (essential appliances, priority, flexibility, peak tariff, comfort, occupancy). They fall into two groups:

**A. Fact-derivation rules** — turn raw sensor/tariff readings into higher-level facts:

| Rule | IF | THEN |
|---|---|---|
| R0 | Peak-hour flag is known | `tariff_status` = "expensive" / "normal" |
| R0b | Occupancy is known | `home_empty` = True / False |
| R0c | Appliance is an AC and temperature is known | `comfort_critical` = True if temp ≥ 38°C |

**B. Decision rules** — turn facts (raw + derived) into a recommendation. They are checked in priority order and the first one whose condition is true "wins" (this is the engine's conflict-resolution strategy):

| Rule | IF | THEN |
|---|---|---|
| R1 | Appliance is essential | Keep it ON — never delay/switch off |
| R2 | AC and `comfort_critical` is True | Keep AC ON despite peak tariff |
| R3 | Priority is High (and not essential/comfort case) | Run as scheduled, but raise a cost alert |
| R4 | Flexible appliance AND `tariff_status` is expensive | Delay to the next off-peak slot |
| R5 | `home_empty` is True and appliance is ON and non-essential | Turn it OFF |
| R6 | None of the above apply | No change needed |

## 2. Choosing the Inference Mechanism

**Forward chaining** is used here.

- The system starts from **known facts** (sensor readings: temperature, occupancy, tariff, appliance state) and **fires every rule whose condition is satisfied**, deriving new facts until nothing new can be derived (a *fixpoint*). This is exactly the "given the current situation, tell me everything relevant" pattern the Optimizer needs — it must react automatically to whatever data currently exists.
- **Backward chaining** would instead start from a *goal* (e.g. "should the washing machine run now?") and work backwards to see if it's provable. That fits a query-driven system, not a system that must scan every appliance and generate recommendations proactively.

Because the Optimizer needs to look at everything it currently knows and generate recommendations for all appliances (data-driven, not goal-driven), **forward chaining is the correct choice**.

# Additional Discussion and Analysis

## 1. Rule Base Summary

The following table summarizes the production rules implemented in the Smart Electricity Consumption Optimizer.

| Rule ID | IF (Condition) | THEN (Action) | Purpose |
|----------|----------------|---------------|----------|
| R0 | Peak Hour = True | Tariff = Expensive | Derive electricity tariff information |
| R1 | Appliance is Essential | Keep Appliance ON | Ensure safety and uninterrupted operation |
| R2 | Temperature > 34°C AND Occupancy = Present | Keep Air Conditioner ON | Maintain user comfort |
| R3 | Occupancy = Absent | Turn OFF unnecessary appliances | Reduce unnecessary energy consumption |
| R4 | Peak Hour = True AND Appliance is Flexible | Delay Appliance | Reduce electricity cost during peak hours |
| R5 | Appliance Priority = High | Do Not Delay Appliance | Preserve important household activities |
| R6 | Off-Peak Hour = True AND Flexible Appliance Waiting | Start Appliance | Shift energy usage to cheaper tariff period |

---

## 2. Rule Evaluation Strategy

The inference engine evaluates rules using **Forward Chaining**. Rules are checked in a logical order to ensure that critical household requirements are satisfied before optimisation decisions are made.

**Rule Priority Order**

1. Derive new facts (Tariff Information)
2. Essential appliance protection
3. User comfort rules
4. Occupancy-based energy saving
5. Appliance priority validation
6. Cost optimisation and scheduling

This priority ensures that safety and comfort are never compromised while reducing electricity consumption.

---

## 3. Forward Chaining Decision Flow

```text
                Sensor Inputs
                     │
                     ▼
           Knowledge Base (Facts)
                     │
                     ▼
         Forward Chaining Engine
                     │
                     ▼
          Match Applicable Rules
                     │
                     ▼
              Fire Matching Rule
                     │
                     ▼
            Update Knowledge Base
                     │
             More Rules Applicable?
               │              │
              Yes             No
               │              │
               └──────► Repeat
                              │
                              ▼
               Generate Final Recommendation
```

---

## 4. Example Decision Path

### Scenario

```
Time = 6:00 PM
Peak Hour = True
Occupancy = Present
Temperature = 35°C
Appliance = Washing Machine
Flexible = Yes
Priority = Medium
```

### Inference Process

```
Facts Loaded
      │
      ▼
Peak Hour Detected
      │
      ▼
Flexible Appliance Identified
      │
      ▼
Priority = Medium
      │
      ▼
Rule R4 Triggered
      │
      ▼
Delay Washing Machine
      │
      ▼
Recommendation Generated
```

### Final Recommendation

```
Delay Washing Machine by 2 Hours
Expected Benefit:
• Reduced electricity cost
• Lower peak-hour load
• No impact on user comfort
```

---

## 5. Future Integration with Machine Learning and Deep Learning

The current rule engine operates using manually defined facts and production rules. In the subsequent phases of the project, Machine Learning and Deep Learning models will provide additional facts to the knowledge base.

For example:

- Predicted Electricity Consumption
- Predicted Peak Demand
- Forecast Weather Conditions
- Estimated Occupancy Pattern

These predicted values will be treated as new facts, allowing the same forward chaining engine to make more intelligent and proactive decisions.

Example:

```
Predicted Consumption > 8 kWh
Peak Hour = True

↓

Recommend delaying all non-essential flexible appliances.
```

Thus, the rule engine acts as the decision-making layer, while the ML and LSTM models act as the prediction layer of the complete intelligent system.

---

## 6. Conclusion

A rule-based decision engine was successfully designed using **Forward Chaining** and a structured set of production rules. The system analyses household facts such as occupancy, weather conditions, appliance characteristics, and electricity tariff to infer energy-saving recommendations. The modular separation of the **Knowledge Base**, **Rule Base**, and **Inference Engine** improves maintainability, scalability, and explainability. Furthermore, this architecture provides a seamless foundation for integrating Machine Learning and LSTM-based electricity consumption predictions in later stages of the Smart Electricity Consumption Optimizer.

In [ ]:
# ---- 3. Rule engine implementation ----

class Rule:
    """A single IF-THEN rule.
    condition: function(facts) -> True/False
    action:    function(facts) -> mutates facts in place
    """
    def __init__(self, rule_id, description, condition, action):
        self.id = rule_id
        self.description = description
        self.condition = condition
        self.action = action


def forward_chain(facts, rules, verbose=True):
    """Runs the classic forward-chaining match-fire loop until no fact changes
    (fixpoint). Returns the final fact base and the ordered list of rules
    that fired (the 'decision path')."""
    facts = dict(facts)          # work on a copy, don't mutate the caller's dict
    decision_path = []
    changed = True

    while changed:
        changed = False
        for rule in rules:
            if rule.condition(facts):
                before = dict(facts)
                rule.action(facts)
                if facts != before:               # only counts as 'firing' if it changed something
                    decision_path.append(rule.id)
                    if verbose:
                        print(f"  -> {rule.id} fired: {rule.description}")
                    changed = True

    return facts, decision_path


In [ ]:
# ---- 4. Rule base ----

rules = [
    # ---- A. Fact-derivation rules ----
    Rule(
        "R0-tariff",
        "Classify current tariff period",
        condition=lambda f: "peak_hour" in f and "tariff_status" not in f,
        action=lambda f: f.update(tariff_status="expensive" if f["peak_hour"] else "normal"),
    ),
    Rule(
        "R0-occupancy",
        "Classify whether the home is empty",
        condition=lambda f: "occupancy" in f and "home_empty" not in f,
        action=lambda f: f.update(home_empty=(f["occupancy"] == 0)),
    ),
    Rule(
        "R0-comfort",
        "Flag high-temperature comfort situations for AC",
        condition=lambda f: f.get("appliance_name") == "Air Conditioner"
                             and "temperature" in f and "comfort_critical" not in f,
        action=lambda f: f.update(comfort_critical=(f["temperature"] >= 38)),
    ),

    # ---- B. Decision rules (priority order = conflict resolution) ----
    Rule(
        "R1-essential",
        "Essential appliances are never delayed or switched off",
        condition=lambda f: f.get("essential") is True and "action" not in f,
        action=lambda f: f.update(action="Keep ON",
                                   reason="Essential appliance - cannot be delayed or switched off, regardless of tariff."),
    ),
    Rule(
        "R2-comfort-override",
        "High temperature overrides peak-hour cost saving for AC",
        condition=lambda f: f.get("comfort_critical") is True and "action" not in f,
        action=lambda f: f.update(action="Keep AC ON despite peak tariff",
                                   reason=f"Temperature {f.get('temperature')} C is high enough that comfort outweighs the peak-hour saving."),
    ),
    Rule(
        "R3-high-priority",
        "High-priority appliances run as scheduled, but raise a cost alert",
        condition=lambda f: f.get("priority") == "High" and not f.get("essential") and "action" not in f,
        action=lambda f: f.update(action="Run as scheduled (alert only)",
                                   reason="High-priority appliance - not delayed, but user is alerted to peak-hour cost."),
    ),
    Rule(
        "R4-flexible-peak",
        "Flexible appliances are rescheduled out of expensive peak-hour slots",
        condition=lambda f: f.get("flexible") is True and f.get("tariff_status") == "expensive" and "action" not in f,
        action=lambda f: f.update(action="Delay to next off-peak slot",
                                   reason="Flexible appliance running during expensive peak tariff - rescheduling saves cost."),
    ),
    Rule(
        "R5-empty-home",
        "Non-essential appliances left running in an empty home are switched off",
        condition=lambda f: f.get("home_empty") is True and f.get("status") == "ON"
                             and not f.get("essential") and "action" not in f,
        action=lambda f: f.update(action="Turn OFF",
                                   reason="Nobody is home and appliance is non-essential."),
    ),
    Rule(
        "R6-default",
        "No optimisation opportunity found",
        condition=lambda f: "action" not in f,
        action=lambda f: f.update(action="No change needed",
                                   reason="No constraint or optimisation rule applies under current conditions."),
    ),
]

print(f"{len(rules)} rules loaded.")


9 rules loaded.


## 5. Testing with Sample Inputs

These reuse the same three scenarios validated in Task 1's knowledge model.

In [ ]:
# ---- Test Case 1: Washing Machine, flexible, peak hour ----

facts_tc1 = {
    "appliance_name": "Washing Machine",
    "essential": False,
    "priority": "Medium",
    "flexible": True,
    "status": "OFF",
    "peak_hour": True,
    "occupancy": 3,
}

print("Test Case 1: Washing Machine during peak hour")
print("Decision path:")
result1, path1 = forward_chain(facts_tc1, rules)
print()
print("Final facts:", result1)
print()
print(f"RECOMMENDATION: {result1['action']}")
print(f"REASON:         {result1['reason']}")


Test Case 1: Washing Machine during peak hour
Decision path:
  -> R0-tariff fired: Classify current tariff period
  -> R0-occupancy fired: Classify whether the home is empty
  -> R4-flexible-peak fired: Flexible appliances are rescheduled out of expensive peak-hour slots

Final facts: {'appliance_name': 'Washing Machine', 'essential': False, 'priority': 'Medium', 'flexible': True, 'status': 'OFF', 'peak_hour': True, 'occupancy': 3, 'tariff_status': 'expensive', 'home_empty': False, 'action': 'Delay to next off-peak slot', 'reason': 'Flexible appliance running during expensive peak tariff - rescheduling saves cost.'}

RECOMMENDATION: Delay to next off-peak slot
REASON:         Flexible appliance running during expensive peak tariff - rescheduling saves cost.


In [ ]:
# ---- Test Case 2: Refrigerator, essential, peak hour ----

facts_tc2 = {
    "appliance_name": "Refrigerator",
    "essential": True,
    "priority": "High",
    "flexible": False,
    "status": "ON",
    "peak_hour": True,
    "occupancy": 3,
}

print("Test Case 2: Refrigerator during peak hour")
print("Decision path:")
result2, path2 = forward_chain(facts_tc2, rules)
print()
print("Final facts:", result2)
print()
print(f"RECOMMENDATION: {result2['action']}")
print(f"REASON:         {result2['reason']}")


Test Case 2: Refrigerator during peak hour
Decision path:
  -> R0-tariff fired: Classify current tariff period
  -> R0-occupancy fired: Classify whether the home is empty
  -> R1-essential fired: Essential appliances are never delayed or switched off

Final facts: {'appliance_name': 'Refrigerator', 'essential': True, 'priority': 'High', 'flexible': False, 'status': 'ON', 'peak_hour': True, 'occupancy': 3, 'tariff_status': 'expensive', 'home_empty': False, 'action': 'Keep ON', 'reason': 'Essential appliance - cannot be delayed or switched off, regardless of tariff.'}

RECOMMENDATION: Keep ON
REASON:         Essential appliance - cannot be delayed or switched off, regardless of tariff.


In [ ]:
# ---- Test Case 3: Air Conditioner, high temperature, peak hour ----

facts_tc3 = {
    "appliance_name": "Air Conditioner",
    "essential": False,
    "priority": "High",
    "flexible": False,
    "status": "ON",
    "peak_hour": True,
    "temperature": 39,
    "occupancy": 3,
}

print("Test Case 3: Air Conditioner, 39 C, peak hour")
print("Decision path:")
result3, path3 = forward_chain(facts_tc3, rules)
print()
print("Final facts:", result3)
print()
print(f"RECOMMENDATION: {result3['action']}")
print(f"REASON:         {result3['reason']}")


Test Case 3: Air Conditioner, 39 C, peak hour
Decision path:
  -> R0-tariff fired: Classify current tariff period
  -> R0-occupancy fired: Classify whether the home is empty
  -> R0-comfort fired: Flag high-temperature comfort situations for AC
  -> R2-comfort-override fired: High temperature overrides peak-hour cost saving for AC

Final facts: {'appliance_name': 'Air Conditioner', 'essential': False, 'priority': 'High', 'flexible': False, 'status': 'ON', 'peak_hour': True, 'temperature': 39, 'occupancy': 3, 'tariff_status': 'expensive', 'home_empty': False, 'comfort_critical': True, 'action': 'Keep AC ON despite peak tariff', 'reason': 'Temperature 39 C is high enough that comfort outweighs the peak-hour saving.'}

RECOMMENDATION: Keep AC ON despite peak tariff
REASON:         Temperature 39 C is high enough that comfort outweighs the peak-hour saving.


In [ ]:
# ---- Bonus Test Case 4: nobody home, non-essential light left on, off-peak ----

facts_tc4 = {
    "appliance_name": "Living Room Light",
    "essential": False,
    "priority": "Low",
    "flexible": False,
    "status": "ON",
    "peak_hour": False,
    "occupancy": 0,
}

print("Test Case 4: Light left on, nobody home")
print("Decision path:")
result4, path4 = forward_chain(facts_tc4, rules)
print()
print("Final facts:", result4)
print()
print(f"RECOMMENDATION: {result4['action']}")
print(f"REASON:         {result4['reason']}")


Test Case 4: Light left on, nobody home
Decision path:
  -> R0-tariff fired: Classify current tariff period
  -> R0-occupancy fired: Classify whether the home is empty
  -> R5-empty-home fired: Non-essential appliances left running in an empty home are switched off

Final facts: {'appliance_name': 'Living Room Light', 'essential': False, 'priority': 'Low', 'flexible': False, 'status': 'ON', 'peak_hour': False, 'occupancy': 0, 'tariff_status': 'normal', 'home_empty': True, 'action': 'Turn OFF', 'reason': 'Nobody is home and appliance is non-essential.'}

RECOMMENDATION: Turn OFF
REASON:         Nobody is home and appliance is non-essential.


## 6. Explaining the Decision Paths

**Test Case 1 — Washing Machine (flexible, peak hour):**
`R0-tariff` fires first and derives `tariff_status = "expensive"` from `peak_hour = True`. `R0-occupancy` derives `home_empty = False`. None of the higher-priority decision rules apply (not essential, not an AC, not High priority), so `R4-flexible-peak` fires: the appliance is flexible and the tariff is expensive → **delay to off-peak**.

**Test Case 2 — Refrigerator (essential, peak hour):**
The derivation rules fire the same way, but because `essential = True`, `R1-essential` matches *before* any other decision rule gets a chance (it's earlier in priority order and the `"action" not in f` guard blocks every rule after the first one that succeeds) → **keep ON**, regardless of the expensive tariff.

**Test Case 3 — Air Conditioner (39°C, peak hour):**
`R0-comfort` derives `comfort_critical = True` since 39 ≥ 38. It isn't essential, so `R1` doesn't fire, but `R2-comfort-override` does, before the peak-hour/flexibility rules are even considered → **keep AC ON despite peak tariff**. If the temperature had been below 38°C, `comfort_critical` would be `False` and the engine would have fallen through to the High-priority rule instead (run as scheduled with a cost alert).

**Test Case 4 — Light, empty home:**
`R0-occupancy` derives `home_empty = True`. It's non-essential, not flexible-relevant, not AC, not High priority, so the engine falls through to `R5-empty-home`, which matches because the light is ON and nobody is home → **turn OFF**.

In every case the *order* in which rules are listed acts as the conflict-resolution strategy: essential-appliance safety comes first, then comfort, then priority, then cost-saving flexibility, then occupancy — matching the constraint hierarchy defined in Task 1.

## Practice tasks (try these yourself)

Once you're comfortable with the code above, try extending it — this is the best way to actually learn how rule engines work rather than just reading the example:

1. **Add a new rule** `R7-humidity`: if `humidity > 80` and the appliance is a dehumidifier, force it ON regardless of tariff. Add the fact and see it fire.
2. **Break the priority order on purpose**: move `R4-flexible-peak` above `R1-essential` in the `rules` list and rerun Test Case 2 — see how the refrigerator recommendation changes. This shows *why* rule order matters for conflict resolution.
3. **Add a counter**: modify `forward_chain` to also return how many passes (`while` loop iterations) it took to reach the fixpoint, and print it for each test case.
4. **Write your own Test Case 5** for a Dishwasher: Medium priority, flexible, NOT peak hour. What should the engine recommend, and which rule should fire? Predict it before running the code, then check.

If you get stuck on any of these, just paste your attempt back to me and I'll help debug it with you rather than just handing you the fix.